In [1]:
#undef __noinline__

# Dot product

This notebook shows the implementation of a parallel dot product on the GPU.

The dot product of two vectors A and B of length N is:

result = A[0] * B[0] + A[1] * B[1] + ... + A[N-1] * B[N-1]

The key challenge on the GPU is that thousands of threads each compute one partial product — then all those partial results must be summed into a single number. That summing step is called **parallel reduction**, and it is the main lesson of this example.

The new syntax here is:

```cpp
__shared__
```

**Normally each thread has its own private memory. Shared memory declares a variable that lives in a special fast memory region shared between all threads in the same block**

```cpp
__syncthreads()
```

**This is a barrier — it forces all threads in the block to stop and wait until every thread has reached that line before any of them can continue**

Bellow is a diagram of how the program 

![Dot Product](images/dot_product_cuda.png)

In [3]:
#define imin(a,b) (a<b?a:b)
#define sum_squares(x) (x*(x+1)*(2*x+1)/6)

const int N = 33 * 1024;
const int threadsPerBlock = 256;
const int blocksPerGrid =

imin( 32, (N+threadsPerBlock-1) / threadsPerBlock );

The `dot` kernel runs on the GPU. Each thread computes partial products
from multiple elements (stride loop), stores its sum in shared memory,
then collaborates with the other threads in its block to reduce everything
down to a single value via parallel reduction.

In [4]:
__global__ void dot( float *a, float *b, float *c ) {
    __shared__ float cache[threadsPerBlock];
    int tid = threadIdx.x + blockIdx.x * blockDim.x;
    int cacheIndex = threadIdx.x;
    float temp = 0;
    while (tid < N) {
        temp += a[tid] * b[tid];
        tid += blockDim.x * gridDim.x;
    }
    // set the cache values
    cache[cacheIndex] = temp;
    
    // synchronize threads in this block
    __syncthreads();
    // for reductions, threadsPerBlock must be a power of 2
    // because of the following code
    int i = blockDim.x/2;
    while (i != 0) {
        if (cacheIndex < i)
        cache[cacheIndex] += cache[cacheIndex + i];
        __syncthreads();
        i /= 2;
    }
    if (cacheIndex == 0) c[blockIdx.x] = cache[0];
}

Allocate the vectors on both CPU and GPU. Notice that `partial_c` only
needs `blocksPerGrid` floats (32), not N — one result per block, not per element.

In [5]:
float *a, *b, c, *partial_c;
float *dev_a, *dev_b, *dev_partial_c;

// allocate memory on the CPU side
a = (float*)malloc( N*sizeof(float) );
b = (float*)malloc( N*sizeof(float) );

partial_c = (float*)malloc( blocksPerGrid*sizeof(float) );

// allocate the memory on the GPU
cudaMalloc( (void**)&dev_a, N*sizeof(float) );
cudaMalloc( (void**)&dev_b, N*sizeof(float) );
cudaMalloc( (void**)&dev_partial_c, blocksPerGrid*sizeof(float) );

Initialize the vectors (`a[i] = i`, `b[i] = 2i`), copy them to the GPU,
then launch the kernel. The result comes back as 32 partial sums in `partial_c`.

In [6]:
// fill in the host memory with data
for (int i=0; i<N; i++) {
    a[i] = i;
    b[i] = i*2;
}
// copy the arrays ‘a’ and ‘b’ to the GPU
cudaMemcpy( dev_a, a, N*sizeof(float), cudaMemcpyHostToDevice );
cudaMemcpy( dev_b, b, N*sizeof(float), cudaMemcpyHostToDevice ); 

dot<<<blocksPerGrid,threadsPerBlock>>>( dev_a, dev_b, dev_partial_c );
// copy the array 'c' back from the GPU to the CPU
cudaMemcpy( partial_c, dev_partial_c, blocksPerGrid*sizeof(float), cudaMemcpyDeviceToHost );

Sum the 32 partial results and verify against the closed-form formula.
Since `a[i]=i` and `b[i]=2i`, the expected answer is `2 * Σi² = 2 * sum_squares(N-1)`.

In [7]:
// finnish up on the CPU side
c = 0;
for (int i=0; i<blocksPerGrid; i++) {
    c += partial_c[i];
}

printf( "Does GPU value %.6g = %.6g?\n", c, 2 * sum_squares( (float)(N - 1) ) );

Does GPU value 2.57236e+13 = 2.57236e+13?


Free all allocated memory on both the GPU and CPU.

In [8]:
// free memory on the GPU side
cudaFree( dev_a );
cudaFree( dev_b );
cudaFree( dev_partial_c );
// free memory on the CPU side
free( a );
free( b );
free( partial_c );